# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okashaahmed2/Flyrankaiintern/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

# **Signal Audit Rationale**:
#**Signal 1 (ctr)**: Testing if lower CTR correlates with higher target decline rate.
#We bucket CTR into quartiles to observe the trend.

#**Signal 2 (impression_tier)**: Testing if high-volume pages contain significant refresh opportunities to justify prioritization.

#V**erdict**: If the data shows clear alignment with our logic, we mark the signal as CONFIRMED.


In [1]:

import pandas as pd
import numpy as np

# 1. Data load
url = "https://raw.githubusercontent.com/okashaahmed2/Flyrankaiintern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['target'] = (df['trend_direction'] == 'down').astype(int)

# 2. Signal 1 Audit: CTR Buckets using pd.cut (Fixing Bin Labels Error)
ctr_bins = [-0.001, 0.01, 0.03, 0.08, 1.0]
ctr_labels = ['Q1_Low (<1%)', 'Q2_MedLow (1-3%)', 'Q3_MedHigh (3-8%)', 'Q4_High (>8%)']

df['ctr_bucket'] = pd.cut(df['ctr'], bins=ctr_bins, labels=ctr_labels)

ctr_audit = df.groupby('ctr_bucket', observed=False)['target'].agg(
    n='count',
    decline_rate='mean'
).reset_index()

print("=== SIGNAL 1 AUDIT: CTR BUCKETS ===")
print(ctr_audit)

# 3. Signal 2 Audit: Impression Tiers vs Decline Rate
imp_audit = df.groupby('impression_tier', observed=False)['target'].agg(
    n='count',
    decline_rate='mean'
).reset_index()

print("\n=== SIGNAL 2 AUDIT: IMPRESSION TIERS ===")
print(imp_audit)


=== SIGNAL 1 AUDIT: CTR BUCKETS ===
          ctr_bucket      n  decline_rate
0       Q1_Low (<1%)  13290      0.498119
1   Q2_MedLow (1-3%)    470      0.731915
2  Q3_MedHigh (3-8%)   1846      0.685265
3      Q4_High (>8%)  12705      0.573475

=== SIGNAL 2 AUDIT: IMPRESSION TIERS ===
  impression_tier      n  decline_rate
0       excellent   1078      0.461967
1            good   7205      0.586121
2             low  11248      0.453947
3        moderate  10469      0.614672


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
import os
import pandas as pd
import numpy as np

# 1. Baseline Score Calculation (Safe Features Only)
# Formula: impressions_90d * (1 - ctr) * (21 - avg_position)
df['baseline_score'] = df['impressions_90d'] * (1 - df['ctr']) * (21 - df['avg_position'].clip(1, 20))

# 2. Rule-Based Assignment (Reason Code & Action Label)
conditions = [
    (df['impression_tier'].isin(['good', 'excellent'])) & (df['ctr'] < 0.03),
    (df['impression_tier'].isin(['good', 'excellent'])) & (df['trend_direction'] == 'down'),
    (df['ctr'] < 0.01)
]

reason_codes = [
    'HIGH_IMP_LOW_CTR',
    'DECLINING_HIGH_TRAFFIC',
    'VERY_LOW_CTR'
]

action_labels = [
    'REFRESH_TITLE_AND_METADATA',
    'FULL_CONTENT_REFRESH',
    'OPTIMIZE_SERP_SNIPPET'
]

df['reason_code'] = np.select(conditions, reason_codes, default='HEALTHY_OR_LOW_PRIORITY')
df['action_label'] = np.select(conditions, action_labels, default='MONITOR_ONLY')

# 3. Sort Ranked Queue by Baseline Score Descending
ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# 4. Save to CSV (work/outputs/baseline_action_score.csv)
os.makedirs('../outputs', exist_ok=True)
os.makedirs('work/outputs', exist_ok=True)

output_cols = ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'ctr', 'avg_position']
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
ranked_queue[output_cols].to_csv('../outputs/baseline_action_score.csv', index=False)

print("--- BASELINE QUEUE CREATED SUCCESSFULLY ---")
print(f"Total Rows Processed: {len(ranked_queue):,}")
print("File Written: work/outputs/baseline_action_score.csv\n")
print("Top 5 Ranked Baseline Items:")
print(ranked_queue[['content_id', 'baseline_score', 'reason_code', 'action_label']].head())


--- BASELINE QUEUE CREATED SUCCESSFULLY ---
Total Rows Processed: 30,000
File Written: work/outputs/baseline_action_score.csv

Top 5 Ranked Baseline Items:
             content_id  baseline_score              reason_code  \
0  content_8c19996aa890     8007987.700   DECLINING_HIGH_TRAFFIC   
1  content_5fe46e04994d     7479946.320   DECLINING_HIGH_TRAFFIC   
2  content_aaef01a50def     6050175.300  HEALTHY_OR_LOW_PRIORITY   
3  content_1a9e894be2e2     5447796.200   DECLINING_HIGH_TRAFFIC   
4  content_4c36c775b818     5109415.399   DECLINING_HIGH_TRAFFIC   

           action_label  
0  FULL_CONTENT_REFRESH  
1  FULL_CONTENT_REFRESH  
2          MONITOR_ONLY  
3  FULL_CONTENT_REFRESH  
4  FULL_CONTENT_REFRESH  


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# Load generated baseline queue
top_10 = pd.read_csv('work/outputs/baseline_action_score.csv').head(10)

print("=== TOP 10 BASELINE RECOMMENDATIONS ===")
for idx, row in top_10.iterrows():
    print(f"Rank {idx+1}: Content ID: {row['content_id']} | Score: {row['baseline_score']:,.0f} | Action: {row['action_label']} | Reason: {row['reason_code']}")
    print(f"        Metrics -> Impressions: {row['impressions_90d']:,} | CTR: {row['ctr']:.2%} | Avg Pos: {row['avg_position']:.1f}\n")






=== TOP 10 BASELINE RECOMMENDATIONS ===
Rank 1: Content ID: content_8c19996aa890 | Score: 8,007,988 | Action: FULL_CONTENT_REFRESH | Reason: DECLINING_HIGH_TRAFFIC
        Metrics -> Impressions: 509,252 | CTR: 15.00% | Avg Pos: 2.5

Rank 2: Content ID: content_5fe46e04994d | Score: 7,479,946 | Action: FULL_CONTENT_REFRESH | Reason: DECLINING_HIGH_TRAFFIC
        Metrics -> Impressions: 517,715 | CTR: 14.00% | Avg Pos: 4.2

Rank 3: Content ID: content_aaef01a50def | Score: 6,050,175 | Action: MONITOR_ONLY | Reason: HEALTHY_OR_LOW_PRIORITY
        Metrics -> Impressions: 517,109 | CTR: 25.00% | Avg Pos: 5.4

Rank 4: Content ID: content_1a9e894be2e2 | Score: 5,447,796 | Action: FULL_CONTENT_REFRESH | Reason: DECLINING_HIGH_TRAFFIC
        Metrics -> Impressions: 416,180 | CTR: 23.00% | Avg Pos: 4.0

Rank 5: Content ID: content_4c36c775b818 | Score: 5,109,415 | Action: FULL_CONTENT_REFRESH | Reason: DECLINING_HIGH_TRAFFIC
        Metrics -> Impressions: 463,103 | CTR: 41.00% | Avg Pos: 2.

#**Rank 1 (content_8c19996aa890 - FULL_CONTENT_REFRESH)**: High volume page sitting at Position 2.5 with 15% CTR. What makes it wrong:
#Full content rewrite risks losing primary keyword rankings that maintain position 2.5.

#**Rank 5 (content_4c36c775b818 - FULL_CONTENT_REFRESH)**: Has an exceptional CTR of 41.0% at Position 2.3. What makes it wrong: Flagged purely due to traffic drop trend.
#High CTR indicates strong title-intent match; drop could be seasonal. A full refresh could destroy CTR.

#**Rank 6 (content_8451fc6f034d - MONITOR_ONLY)**: Low CTR (3.0%) despite strong Position 2.3. What makes it wrong: Assigned 'MONITOR_ONLY'
#because it lacks a downward trend, yet a 3% CTR at position 2 indicates a major title/meta mismatch opportunity missed.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

#**Weak Picks & Rule Edge Cases**:
#**False Positive Risk**: Pages with exceptionally high CTR (>30%) that experience seasonal traffic drops are mistakenly assigned FULL_CONTENT_REFRESH.
#Rewriting content on high-intent pages causes revenue loss.


#**Fix for ML Stage (Week 5)**: The future Machine Learning model must learn non-linear interactions between ctr and
#trend_direction rather than relying on hardcoded IF-ELSE rules.



# Weak Picks Identification (High CTR but flagged for refresh)


In [4]:

weak_picks = ranked_queue[
    (ranked_queue['action_label'] == 'FULL_CONTENT_REFRESH') &
    (ranked_queue['ctr'] > 0.30)
].head(5)

print("=== WEAK PICKS / RISKY RECOMMENDATIONS ===")
print(f"Found {len(weak_picks)} high-risk rows where CTR > 30% but action is FULL_CONTENT_REFRESH:")
print(weak_picks[['content_id', 'baseline_score', 'ctr', 'avg_position', 'action_label']])

=== WEAK PICKS / RISKY RECOMMENDATIONS ===
Found 5 high-risk rows where CTR > 30% but action is FULL_CONTENT_REFRESH:
              content_id  baseline_score   ctr  avg_position  \
4   content_4c36c775b818     5109415.399  0.41           2.3   
12  content_2c2606c5d176     2743062.504  0.53           4.2   
39  content_01908772c6db     1756799.550  0.45           4.0   
75  content_cbd93118300b     1388700.936  0.46           3.3   
77  content_53f466b9f954     1382938.692  0.33           4.6   

            action_label  
4   FULL_CONTENT_REFRESH  
12  FULL_CONTENT_REFRESH  
39  FULL_CONTENT_REFRESH  
75  FULL_CONTENT_REFRESH  
77  FULL_CONTENT_REFRESH  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.